<center>
<img src="../../img/ods_stickers.jpg">
    
## [mlcourse.ai](https://mlcourse.ai) - دورة التعلم الآلي المفتوحة
المؤلف: [يوري كاشنيتسكي](https://www.linkedin.com/in/festline/). يتم توزيع كل المحتوى بموجب ترخيص [Creative Commons CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/).



## <center> المهمة 4 (تجريبي). الحل
### <center> كشف السخرية مع الانحدار اللوجستي
    
** نفس المهمة مثل [Kaggle Kernel](https://www.kaggle.com/kashnitsky/a4-demo-sarcasm-detection-with-logit) + [الحل](https://www.kaggle.com/kashnitsky/a4-demo-sarcasm-detection-with-logit-solution).**
سنستخدم مجموعة البيانات من [الورقة](https://arxiv.org/abs/1704.05579) "مجموعة كبيرة من التعليقات التوضيحية للسخرية" مع أكثر من مليون تعليق من Reddit، مصنفة على أنها ساخرة أم لا. يمكن العثور على نسخة تمت معالجتها على Kaggle في شكل [Kaggle Dataset](https://www.kaggle.com/danofer/sarcasm).


In [ ]:
PATH_TO_DATA = "../input/sarcasm/train-balanced-sarcasm.csv"

In [ ]:
# some necessary imports
import os

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [ ]:
train_df = pd.read_csv(PATH_TO_DATA)

In [ ]:
train_df.head()

In [ ]:
train_df.info()


بعض التعليقات مفقودة، لذلك نقوم بإسقاط الصفوف المقابلة.


In [ ]:
train_df.dropna(subset=["comment"], inplace=True)


نلاحظ أن مجموعة البيانات متوازنة بالفعل


In [ ]:
train_df["label"].value_counts()


نقوم بتقسيم البيانات إلى أجزاء للتدريب والتحقق.


In [ ]:
train_texts, valid_texts, y_train, y_valid = train_test_split(
    train_df["comment"], train_df["label"], random_state=17
)


## المهام:
1. قم بتحليل مجموعة البيانات، وقم بعمل بعض المخططات. قد يكون هذا [Kernel](https://www.kaggle.com/sudalairajkumar/simple-exploration-notebook-qiqc) بمثابة مثال
2. قم ببناء خط أنابيب Tf-Idf + الانحدار اللوجستي للتنبؤ بالسخرية (`label`) بناءً على نص التعليق على Reddit (`comment`).
3. ارسم الكلمات/الرموز الكبيرة التي تتنبأ بالسخرية بشكل أكبر (يمكنك استخدام [eli5](https://github.com/TeamHG-Memex/eli5) لذلك)
4. (اختياريًا) قم بإضافة subreddits كميزات جديدة لتحسين أداء النموذج. قم بتطبيق نهج حقيبة الكلمات هنا، أي تعامل مع كل subreddit كميزة جديدة.



### الجزء الأول. تحليل البيانات الاستكشافية



توزيع أطوال التعليقات الساخرة والعادية هو نفسه تقريبًا.


In [ ]:
train_df.loc[train_df["label"] == 1, "comment"].str.len().apply(np.log1p).hist(
    label="sarcastic", alpha=0.5
)
train_df.loc[train_df["label"] == 0, "comment"].str.len().apply(np.log1p).hist(
    label="normal", alpha=0.5
)
plt.legend();

In [ ]:
from wordcloud import STOPWORDS, WordCloud

In [ ]:
wordcloud = WordCloud(
    background_color="black",
    stopwords=STOPWORDS,
    max_words=200,
    max_font_size=100,
    random_state=17,
    width=800,
    height=400,
)


سحابة الكلمات جميلة، ولكنها ليست مفيدة جدًا


In [ ]:
plt.figure(figsize=(16, 12))
wordcloud.generate(str(train_df.loc[train_df["label"] == 1, "comment"]))
plt.imshow(wordcloud);

In [ ]:
plt.figure(figsize=(16, 12))
wordcloud.generate(str(train_df.loc[train_df["label"] == 0, "comment"]))
plt.imshow(wordcloud);


دعونا نحلل ما إذا كانت بعض المنتديات الفرعية أكثر "سخرية" في المتوسط من غيرها


In [ ]:
sub_df = train_df.groupby("subreddit")["label"].agg([np.size, np.mean, np.sum])
sub_df.sort_values(by="sum", ascending=False).head(10)

In [ ]:
sub_df[sub_df["size"] > 1000].sort_values(by="mean", ascending=False).head(10)

الشيء نفسه بالنسبة للمؤلفين لا يعطي الكثير من المعرفة. باستثناء حقيقة أنه تم أخذ عينات من تعليقات شخص ما - يمكننا أن نرى نفس الكم من التعليقات الساخرة وغير الساخرة.


In [ ]:
sub_df = train_df.groupby("author")["label"].agg([np.size, np.mean, np.sum])
sub_df[sub_df["size"] > 300].sort_values(by="mean", ascending=False).head(10)

In [ ]:
sub_df = (
    train_df[train_df["score"] >= 0]
    .groupby("score")["label"]
    .agg([np.size, np.mean, np.sum])
)
sub_df[sub_df["size"] > 300].sort_values(by="mean", ascending=False).head(10)

In [ ]:
sub_df = (
    train_df[train_df["score"] < 0]
    .groupby("score")["label"]
    .agg([np.size, np.mean, np.sum])
)
sub_df[sub_df["size"] > 300].sort_values(by="mean", ascending=False).head(10)


### الجزء الثاني. تدريب النموذج


In [ ]:
# build bigrams, put a limit on maximal number of features
# and minimal word frequency
tf_idf = TfidfVectorizer(ngram_range=(1, 2), max_features=50000, min_df=2)
# multinomial logistic regression a.k.a softmax classifier
logit = LogisticRegression(C=1, n_jobs=4, solver="lbfgs", random_state=17, verbose=1)
# sklearn's pipeline
tfidf_logit_pipeline = Pipeline([("tf_idf", tf_idf), ("logit", logit)])

In [ ]:
%%time
tfidf_logit_pipeline.fit(train_texts, y_train)

In [ ]:
%%time
valid_pred = tfidf_logit_pipeline.predict(valid_texts)

In [ ]:
accuracy_score(y_valid, valid_pred)


### الجزء الثالث. شرح النموذج


In [ ]:
def plot_confusion_matrix(
    actual,
    predicted,
    classes,
    normalize=False,
    title="Confusion matrix",
    figsize=(7, 7),
    cmap=plt.cm.Blues,
    path_to_save_fig=None,
):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    import itertools

    from sklearn.metrics import confusion_matrix

    cm = confusion_matrix(actual, predicted).T
    if normalize:
        cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=figsize)
    plt.imshow(cm, interpolation="nearest", cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=90)
    plt.yticks(tick_marks, classes)

    fmt = ".2f" if normalize else "d"
    thresh = cm.max() / 2.0
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(
            j,
            i,
            format(cm[i, j], fmt),
            horizontalalignment="center",
            color="white" if cm[i, j] > thresh else "black",
        )

    plt.tight_layout()
    plt.ylabel("Predicted label")
    plt.xlabel("True label")

    if path_to_save_fig:
        plt.savefig(path_to_save_fig, dpi=300, bbox_inches="tight")


مصفوفة الارتباك متوازنة تماما.


In [ ]:
plot_confusion_matrix(
    y_valid,
    valid_pred,
    tfidf_logit_pipeline.named_steps["logit"].classes_,
    figsize=(8, 8),
)


وبالفعل يمكننا التعرف على بعض العبارات الدالة على السخرية. مثل "نعم بالتأكيد". 


In [ ]:
import eli5

eli5.show_weights(
    estimator=tfidf_logit_pipeline.named_steps["logit"],
    vec=tfidf_logit_pipeline.named_steps["tf_idf"],
)


لذا فإن اكتشاف السخرية أمر سهل.
<img src="@@KEEP_00011@@ />



### الجزء الرابع. تحسين النموذج


In [ ]:
subreddits = train_df["subreddit"]
train_subreddits, valid_subreddits = train_test_split(subreddits, random_state=17)


سيكون لدينا متجهات Tf-Idf منفصلة للتعليقات وللمحتوى الفرعي. من الممكن أيضًا الالتزام بخط أنابيب، لكن في هذه الحالة يصبح الأمر أقل وضوحًا. [مثال](https://stackoverflow.com/questions/36731813/computing-separate-tfidf-scores-for-two-different-columns-using-sklearn)


In [ ]:
tf_idf_texts = TfidfVectorizer(ngram_range=(1, 2), max_features=50000, min_df=2)
tf_idf_subreddits = TfidfVectorizer(ngram_range=(1, 1))


قم بإجراء التحولات بشكل منفصل للتعليقات وsubreddits. 


In [ ]:
%%time
X_train_texts = tf_idf_texts.fit_transform(train_texts)
X_valid_texts = tf_idf_texts.transform(valid_texts)

In [ ]:
X_train_texts.shape, X_valid_texts.shape

In [ ]:
%%time
X_train_subreddits = tf_idf_subreddits.fit_transform(train_subreddits)
X_valid_subreddits = tf_idf_subreddits.transform(valid_subreddits)

In [ ]:
X_train_subreddits.shape, X_valid_subreddits.shape


ثم قم بتجميع كافة الميزات معًا.


In [ ]:
from scipy.sparse import hstack

X_train = hstack([X_train_texts, X_train_subreddits])
X_valid = hstack([X_valid_texts, X_valid_subreddits])

In [ ]:
X_train.shape, X_valid.shape


تدريب نفس الانحدار اللوجستي.


In [ ]:
logit.fit(X_train, y_train)

In [ ]:
%%time
valid_pred = logit.predict(X_valid)

In [ ]:
accuracy_score(y_valid, valid_pred)


كما نرى، زادت الدقة قليلاً.



## الروابط:
  - مكتبة التعلم الآلي [Scikit-learn](https://scikit-learn.org/stable/index.html) (المعروف أيضًا باسم sklearn)
  - النواة على [الانحدار اللوجستي](https://www.kaggle.com/kashnitsky/topic-4-linear-models-part-2-classification) وتطبيقاتها على [تصنيف النص](https://www.kaggle.com/kashnitsky/topic-4-linear-models-part-4-more-of-logit)، وكذلك [النواة](https://www.kaggle.com/kashnitsky/topic-6-feature-engineering-and-feature-selection) في هندسة الميزات واختيار الميزات
  - [Kaggle Kernel](https://www.kaggle.com/abhishek/approaching-almost-any-nlp-problem-on-kaggle) "نقترب (تقريبًا) من أي مشكلة في البرمجة اللغوية العصبية على Kaggle"
  - [ELI5](https://github.com/TeamHG-Memex/eli5) لشرح توقعات النماذج